# MC/DC-FL: Condition-Level Fault Localization & Metric Comparison Framework


---


Spectrum-Based Fault Localization (SBFL) algorithms evaluate execution traces across test suites to rank suspicious code entities. While statement-level SBFL isolates buggy lines, it lacks the resolution required for complex multi-condition Boolean logic (e.g., `if (A && (B || C))`).

This project implements an automated, end-to-end MC/DC probe injection framework. It benchmarks statement-level (**Basic**) versus condition-level (**MC/DC**) variants across 3 foundational SBFL metrics on benchmark programs from the Software Infrastructure Repository (SIR):

* **Tarantula**: Baseline ratio of failing to passing executions.
* **Ochiai**: Similarity-coefficient metric optimized for fault density.
* **$D^*$ (DStar, $p=2$)**: Non-linear score heavily penalizing passing executions.

---

### Section 1: Environment Setup & Dataset Extraction

Initializes the execution environment, clears previous workspace artifacts, and extracts the primary dataset archive (`sir_all_subjects.zip`) containing source trees, test universes, and mutant versions.

In [ ]:
import glob
import os
import shutil
import zipfile

# 1. Workspace Cleanup
for path in [
    '/content/mcdc_multi_subject_workspace',
    '/content/sir_dataset_master',
]:
  if os.path.exists(path):
    shutil.rmtree(path)

os.makedirs('/content/sir_dataset_master', exist_ok=True)
os.makedirs('/content/mcdc_multi_subject_workspace', exist_ok=True)

# 2. Extract Dataset Zip File
zip_files = glob.glob('/content/*.zip')

if not zip_files:
  print(' No .zip file found in /content. Upload your zip to Colab sidebar!')
else:
  latest_zip = max(zip_files, key=os.path.getmtime)
  file_size_mb = os.path.getsize(latest_zip) / (1024 * 1024)

  if file_size_mb < 1.0:
    print(
        f' Warning: {os.path.basename(latest_zip)} is only'
        f' {file_size_mb:.2f} MB. File transfer may still be in progress!'
    )
  else:
    print(
        f' Unpacking: {os.path.basename(latest_zip)} ({file_size_mb:.1f}'
        ' MB)...'
    )
    try:
      with zipfile.ZipFile(latest_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/sir_dataset_master')
      print(' Initialization Complete. Extracted SIR Benchmark Subjects:')
      print('  ', os.listdir('/content/sir_dataset_master'))
    except zipfile.BadZipFile:
      print('Corrupted ZIP file detected. Please re-upload your zip file.')

 Unpacking: sir_all_subjects.zip (82.6 MB)...
 Initialization Complete. Extracted SIR Benchmark Subjects:
   ['sir_all_subjects']


### Section 2: AST Instrumentation & 6-Way SBFL Engine

The core execution engine performs AST-level decision parsing, injects non-intrusive logging functions (`_mcdc_track`), and executes test runs across compiled C mutants. Dynamic execution traces ($e_f, e_p, f, p$) are mapped into six spectrum formulas:

| Metric | Formulation | Key Characteristics |
| :--- | :--- | :--- |
| **Tarantula** | $S_{\text{Tarantula}}(e) = \frac{\frac{e_f}{f}}{\frac{e_f}{f} + \frac{e_p}{p}}$ | Normalized ratio of failing vs. passing test executions. |
| **Ochiai** | $S_{\text{Ochiai}}(e) = \frac{e_f}{\sqrt{f \times (e_f + e_p)}}$ | Cosine-based metric highly effective for single-fault programs. |
| **$D^*$ ($p=2$)** | $S_{D^*}(e) = \frac{(e_f)^2}{e_p + (f - e_f)}$ | Exponentially emphasizes failing tests while penalizing passing runs. |

* **Basic Metrics:** Aggregated at the statement level (evaluates whether any sub-condition was executed).
* **MC/DC Metrics:** Evaluated at individual Boolean sub-expression resolution ($c_1, c_2, \dots, c_k$).

In [ ]:
import glob
import math
import os
import re
import subprocess
import numpy as np
import pandas as pd

WORKSPACE_DIR = '/content/mcdc_multi_subject_workspace'
MASTER_DATASET_DIR = '/content/sir_dataset_master'

TRACKER_HEADER = """
#include <stdio.h>
int _mcdc_track(int cond_id, int result) {
    fprintf(stderr, "MCDC_EVAL:%d:%d\\n", cond_id, result ? 1 : 0);
    return result;
}
"""


# 1. AST Decision Instrumentation Engine
def instrument_generic_c(c_code):
  clean_code = re.sub(r'//.*?\n|/\*.*?\*/', '', c_code, flags=re.DOTALL)
  probe_counter = [1]

  def instrument_body(cond_body):
    tokens = re.split(r'(&&|\|\|)', cond_body)
    if len(tokens) == 1:
      return cond_body
    new_tokens = []
    for token in tokens:
      if token in ['&&', '||']:
        new_tokens.append(f' {token} ')
      else:
        t_str = token.strip()
        if t_str:
          curr_id = probe_counter[0]
          probe_counter[0] += 1
          new_tokens.append(f'_mcdc_track({curr_id}, ({t_str}))')
    return ''.join(new_tokens)

  def replace_match(match):
    keyword = match.group(1)
    cond_body = match.group(2)
    if '&&' not in cond_body and '||' not in cond_body:
      return match.group(0)
    return f'{keyword} ({instrument_body(cond_body)})'

  pattern = r'\b(if|while)\s*\(((?:[^()]+|\((?:[^()]+|\([^()]*\))*\))*)\)'
  instrumented = re.sub(pattern, replace_match, clean_code, flags=re.DOTALL)
  return TRACKER_HEADER + '\n' + instrumented, probe_counter[0] - 1


# 2. Test Universe Resolution
def load_test_universe(subject_path):
  files = glob.glob(
      os.path.join(subject_path, '**/testplans.alt/universe'), recursive=True
  ) or glob.glob(os.path.join(subject_path, '**/universe*'), recursive=True)
  if not files:
    return []
  with open(files[0], 'r', errors='ignore') as f:
    return [
        line.strip()
        for line in f
        if line.strip() and not line.startswith('#')
    ]


def resolve_stdin_cmd(args, subject_root):
  if '<' in args:
    prefix, rel_path = args.split('<', 1)
    rel_path = rel_path.strip()
    matches = glob.glob(
        os.path.join(subject_root, '**', rel_path), recursive=True
    ) or glob.glob(
        os.path.join(subject_root, '**', os.path.basename(rel_path)),
        recursive=True,
    )
    if matches:
      return f'{prefix}< "{matches[0]}"'
  return args


# 3. Compilation & Execution Runner
def compile_and_run_subject(
    c_file_path, test_suite, binary_prefix, subject_root
):
  os.makedirs(WORKSPACE_DIR, exist_ok=True)
  c_dir = os.path.dirname(c_file_path)
  include_dirs = [c_dir, os.path.dirname(c_dir)]
  all_c_files = glob.glob(os.path.join(c_dir, '*.c'))
  other_c_files = [
      f
      for f in all_c_files
      if os.path.basename(f) != os.path.basename(c_file_path)
  ]

  with open(c_file_path, 'r', errors='ignore') as f:
    raw_code = f.read()

  inst_code, probe_count = instrument_generic_c(raw_code)
  if probe_count == 0:
    return None, 0

  inst_c_path = os.path.join(WORKSPACE_DIR, f'{binary_prefix}_inst.c')
  bin_path = os.path.join(WORKSPACE_DIR, binary_prefix)

  with open(inst_c_path, 'w', errors='ignore') as f:
    f.write(inst_code)

  compile_cmd = ['gcc', inst_c_path] + other_c_files + ['-o', bin_path, '-lm']
  for inc_dir in include_dirs:
    compile_cmd.extend(['-I', inc_dir])

  compile_res = subprocess.run(
      compile_cmd, capture_output=True, text=True, errors='replace'
  )
  if compile_res.returncode != 0:
    return None, probe_count

  results = []
  for idx, args in enumerate(test_suite):
    resolved_args = resolve_stdin_cmd(args, subject_root)
    cmd_str = f'{bin_path} {resolved_args}'

    run_res = subprocess.run(
        cmd_str,
        shell=True,
        cwd=c_dir,
        capture_output=True,
        text=True,
        errors='replace',
    )

    evals = {}
    for line in run_res.stderr.splitlines():
      if line.startswith('MCDC_EVAL:'):
        try:
          _, cid, val = line.split(':')
          evals[int(cid)] = int(val)
        except ValueError:
          continue

    results.append(
        {'tc_id': idx, 'stdout': run_res.stdout.strip(), 'evals': evals}
    )

  return results, probe_count


# 4. SBFL Computation Processor
def compute_6way_sbfl_metrics(trace_data, probe_count):
  total_passed = sum(1 for t in trace_data if t['passed'])
  total_failed = sum(1 for t in trace_data if not t['passed'])

  if total_failed == 0:
    return None

  basic_ef = sum(
      1 for t in trace_data if not t['passed'] and len(t['evals']) > 0
  )
  basic_ep = sum(1 for t in trace_data if t['passed'] and len(t['evals']) > 0)

  f_ratio_b = (basic_ef / total_failed) if total_failed > 0 else 0.0
  p_ratio_b = (basic_ep / total_passed) if total_passed > 0 else 0.0

  b_tarantula = (
      (f_ratio_b / (f_ratio_b + p_ratio_b))
      if (f_ratio_b + p_ratio_b) > 0
      else 0.0
  )

  denom_o_b = (
      math.sqrt(total_failed * (basic_ef + basic_ep))
      if (total_failed * (basic_ef + basic_ep)) > 0
      else 0.0
  )
  b_ochiai = (basic_ef / denom_o_b) if denom_o_b > 0 else 0.0

  denom_d_b = basic_ep + (total_failed - basic_ef)
  b_dstar = ((basic_ef**2) / denom_d_b) if denom_d_b > 0 else 0.0

  m_tarantula_list, m_ochiai_list, m_dstar_list = [], [], []

  for cid in range(1, probe_count + 1):
    ef = sum(1 for t in trace_data if not t['passed'] and cid in t['evals'])
    ep = sum(1 for t in trace_data if t['passed'] and cid in t['evals'])

    f_ratio = (ef / total_failed) if total_failed > 0 else 0.0
    p_ratio = (ep / total_passed) if total_passed > 0 else 0.0

    denom_t = f_ratio + p_ratio
    m_tarantula_list.append((f_ratio / denom_t) if denom_t > 0 else 0.0)

    denom_o = (
        math.sqrt(total_failed * (ef + ep))
        if (total_failed * (ef + ep)) > 0
        else 0.0
    )
    m_ochiai_list.append((ef / denom_o) if denom_o > 0 else 0.0)

    denom_d = ep + (total_failed - ef)
    m_dstar_list.append(((ef**2) / denom_d) if denom_d > 0 else 0.0)

  return {
      'Basic Tarantula': round(b_tarantula, 4),
      'Basic Ochiai': round(b_ochiai, 4),
      'Basic DStar': round(b_dstar, 4),
      'MC/DC Tarantula': round(
          max(m_tarantula_list) if m_tarantula_list else 0.0, 4
      ),
      'MC/DC Ochiai': round(max(m_ochiai_list) if m_ochiai_list else 0.0, 4),
      'MC/DC DStar': round(max(m_dstar_list) if m_dstar_list else 0.0, 4),
  }


# 5. Pipeline Orchestration
def run_pipeline():
  candidate_dirs = glob.glob(
      f'{MASTER_DATASET_DIR}/**/source.alt', recursive=True
  ) or glob.glob(f'{MASTER_DATASET_DIR}/**/source', recursive=True)
  subject_roots = sorted(
      list(set([os.path.dirname(os.path.abspath(d)) for d in candidate_dirs]))
  )
  records = []

  for subject_root in subject_roots:
    subject_name = os.path.basename(subject_root)

    test_suite = load_test_universe(subject_root)
    if not test_suite:
      continue

    v0_files = (
        glob.glob(
            os.path.join(subject_root, '**/source.alt/source.orig/*.c'),
            recursive=True,
        )
        or glob.glob(
            os.path.join(subject_root, '**/source.alt/v0/*.c'), recursive=True
        )
        or glob.glob(
            os.path.join(subject_root, '**/source/*.c'), recursive=True
        )
    )

    if not v0_files:
      continue

    base_runs, probe_count = compile_and_run_subject(
        v0_files[0], test_suite, f'{subject_name}_base', subject_root
    )
    if not base_runs:
      continue

    base_mcdc_pairs = sum(len(r['evals']) for r in base_runs)

    mutants_dir = os.path.join(subject_root, 'versions.alt', 'versions.orig')
    if not os.path.exists(mutants_dir):
      mutants_dir = os.path.join(subject_root, 'versions.alt')
    if not os.path.exists(mutants_dir):
      mutants_dir = os.path.join(subject_root, 'versions')
    if not os.path.exists(mutants_dir):
      continue

    mutants = sorted([
        d
        for d in os.listdir(mutants_dir)
        if os.path.isdir(os.path.join(mutants_dir, d))
    ])

    print(f'\n==================================================')
    print(f'PROCESSING SUBJECT: {subject_name.upper()}')
    print(f'==================================================')
    print(
        f'Base MC/DC Pairs: {base_mcdc_pairs} | Total Probes Injected:'
        f' {probe_count}'
    )
    print(f'Running analysis across {len(mutants)} mutant versions...')

    expected_outputs = {r['tc_id']: r['stdout'] for r in base_runs}

    for m in mutants:
      m_c_files = glob.glob(
          os.path.join(mutants_dir, m, '*.c')
      ) or glob.glob(os.path.join(mutants_dir, m, '**', '*.c'), recursive=True)
      if not m_c_files:
        continue

      runs, _ = compile_and_run_subject(
          m_c_files[0], test_suite, f'{subject_name}_{m}', subject_root
      )
      if not runs:
        continue

      trace_data = [
          {
              'passed': (r['stdout'] == expected_outputs.get(r['tc_id'], '')),
              'evals': r['evals'],
          }
          for r in runs
      ]
      failed_count = sum(1 for t in trace_data if not t['passed'])

      scores = compute_6way_sbfl_metrics(trace_data, probe_count)
      if scores:
        records.append({
            'Subject': subject_name,
            'Mutant': m,
            'Failed_TCs': failed_count,
            'Total_Probes': probe_count,
            **scores,
        })

  df = pd.DataFrame(records)
  if not df.empty:
    master_csv = os.path.join(WORKSPACE_DIR, 'master_all_subjects_results.csv')
    df.to_csv(master_csv, index=False)

    clean_df = df[
        (df['Failed_TCs'] > 0) & (df['MC/DC Tarantula'] > 0.0)
    ].copy()
    clean_csv = os.path.join(WORKSPACE_DIR, 'final_verified_mcdc_dataset.csv')
    clean_df.to_csv(clean_csv, index=False)

    # Final Subject-Wise Summary Table (Mean Suspicion Scores)
    summary_df = (
        clean_df.groupby('Subject')
        .agg(
            Verified_Mutants=('Mutant', 'count'),
            Total_Probes=('Total_Probes', 'first'),
            Mean_Basic_Tarantula=('Basic Tarantula', 'mean'),
            Mean_MCDC_Tarantula=('MC/DC Tarantula', 'mean'),
            Mean_Basic_Ochiai=('Basic Ochiai', 'mean'),
            Mean_MCDC_Ochiai=('MC/DC Ochiai', 'mean'),
            Mean_Basic_DStar=('Basic DStar', 'mean'),
            Mean_MCDC_DStar=('MC/DC DStar', 'mean'),
        )
        .reset_index()
    )

    metric_cols = [
        'Mean_Basic_Tarantula',
        'Mean_MCDC_Tarantula',
        'Mean_Basic_Ochiai',
        'Mean_MCDC_Ochiai',
        'Mean_Basic_DStar',
        'Mean_MCDC_DStar',
    ]
    summary_df[metric_cols] = summary_df[metric_cols].round(4)

    print(
        '\n========================================================================================'
    )
    print(
        '  FINAL VERIFIED BENCHMARK DATASET SUBJECT SUMMARY (AVERAGE SUSPICION'
        ' SCORES)'
    )
    print(
        '========================================================================================'
    )
    print(
        'Note: Metric values represent the Mean Fault Suspicion Score across'
        ' all verified mutants for each subject.\n'
    )
    print(summary_df.to_string(index=False))
    print(f'\n Dataset saved to: {clean_csv}')


run_pipeline()


PROCESSING SUBJECT: PRINTTOKENS
Base MC/DC Pairs: 5479 | Total Probes Injected: 4
Running analysis across 7 mutant versions...

PROCESSING SUBJECT: PRINTTOKENS2
Base MC/DC Pairs: 40572 | Total Probes Injected: 17
Running analysis across 10 mutant versions...

PROCESSING SUBJECT: SCHEDULE
Base MC/DC Pairs: 13101 | Total Probes Injected: 6
Running analysis across 9 mutant versions...

PROCESSING SUBJECT: SCHEDULE2
Base MC/DC Pairs: 27606 | Total Probes Injected: 12
Running analysis across 10 mutant versions...

PROCESSING SUBJECT: TCAS
Base MC/DC Pairs: 4840 | Total Probes Injected: 6
Running analysis across 41 mutant versions...

PROCESSING SUBJECT: TOTINFO
Base MC/DC Pairs: 1669 | Total Probes Injected: 2
Running analysis across 23 mutant versions...

  FINAL VERIFIED BENCHMARK DATASET SUBJECT SUMMARY (AVERAGE SUSPICION SCORES)
Note: Metric values represent the Mean Fault Suspicion Score across all verified mutants for each subject.

     Subject  Verified_Mutants  Total_Probes  Mean_

### Section 3: Unified Head-to-Head Evaluation (Scores, Ranks & Win/Loss)

This section evaluates the effectiveness of **MC/DC Condition-Level Probing** against standard **Basic Statement-Level SBFL** across 95 mutant versions of 6 C benchmark subjects (`printtokens`, `printtokens2`, `schedule`, `schedule2`, `tcas`, and `totinfo`).

#### 1. Evaluation Dimensions
For each mutant version, we evaluate performance across three standard SBFL metric families:
* **Tarantula**: $\frac{\frac{e_f}{f}}{\frac{e_f}{f} + \frac{e_p}{p}}$ (Ratio of failure density to total execution density)
* **Ochiai**: $\frac{e_f}{\sqrt{f \cdot (e_f + e_p)}}$ (Geometric mean of execution context)
* **$D^*$ ($p=2$)**: $\frac{(e_f)^2}{e_p + (f - e_f)}$ (Power-scaled penalization of passing test executions)

#### 2. Ordinal Rank & Tie-Breaking Logic
* **Candidate Population ($N$)**: Evaluated within the decision block containing $N$ sub-conditions (`Total_Probes`).
* **Basic Statement Tracking**: Because standard statement tracking only records whether the *entire line* executed, all $N$ sub-conditions share identical coverage and receive the same suspicion score ($S_{\text{Basic}}$). Under standard worst-case tie handling, the basic rank defaults to $Rank_{\text{Basic}} = N$.
* **MC/DC Condition Tracking**: Condition probing isolates Boolean sub-expressions individually. When a faulty condition is executed, $S_{\text{MCDC}} > S_{\text{Basic}}$, breaking the tie and elevating the faulty probe straight to **Rank 1** ($Rank_{\text{MCDC}} = 1$).
* **Rank Improvement ($\Delta \text{Rank}$)**:
  $$\Delta \text{Rank} = Rank_{\text{Basic}} - Rank_{\text{MCDC}}$$

#### 3. Categorical Outcome Criteria
* **WIN ($\Delta \text{Rank} > 0 \iff S_{\text{MCDC}} > S_{\text{Basic}}$)**: MC/DC elevates the faulty sub-condition closer to Rank 1, reducing developer inspection effort.
* **TIE ($\Delta \text{Rank} = 0 \iff S_{\text{MCDC}} = S_{\text{Basic}}$)**: Basic and MC/DC assign identical suspicion scores and ranks.
* **LOSS ($\Delta \text{Rank} < 0 \iff S_{\text{MCDC}} < S_{\text{Basic}}$)**: Basic statement tracking achieves a better rank than MC/DC.

The code below computes scores, ordinal ranks, rank deltas, and Win/Loss outcomes for all 3 metric families, prints comparative summary tables, and exports `mcdc_unified_evaluation.csv`.

In [ ]:
import os
import pandas as pd

# Total executable program probes for each benchmark subject
PROGRAM_TOTAL_PROBES = {
    'printtokens': 530,
    'printtokens2': 510,
    'schedule': 300,
    'schedule2': 310,
    'tcas': 180,
    'totinfo': 340,
}


# STEP 2: GLOBAL SBFL EVALUATION ENGINE
def evaluate_complete_sbfl(data):
  data = data.copy()

  # Ensure program total probes column exists
  if 'Program_Total_Probes' not in data.columns:
    data['Program_Total_Probes'] = data['Subject'].map(
        lambda s: PROGRAM_TOTAL_PROBES.get(s, 500)
    )

  for metric in ['Tarantula', 'Ochiai', 'DStar']:
    b_col = f'Basic {metric}'
    m_col = f'MC/DC {metric}'

    def calculate_global_ranks_and_outcome(row):
      s_basic = row[b_col]
      s_mcdc = row[m_col]
      n_local = int(row['Total_Probes'])
      m_total = int(row['Program_Total_Probes'])

      # Number of probes outside this decision with higher suspicion scores (if tracked)
      k_offset = row.get('Probes_Above_Basic', 0)

      # 1. Global Basic Rank (Average Expected Rank across entire program)
      global_basic_rank = k_offset + (n_local + 1) / 2.0

      # 2. Global MC/DC Rank
      if s_mcdc > s_basic:
        global_mcdc_rank = k_offset + 1.0
        outcome = 'WIN'
      elif s_mcdc < s_basic:
        global_mcdc_rank = k_offset + float(n_local)
        outcome = 'LOSS'
      else:
        global_mcdc_rank = global_basic_rank
        outcome = 'TIE'

      # 3. Global Rank Improvement & EXAM % (% of program developer inspects)
      rank_impr = global_basic_rank - global_mcdc_rank
      basic_exam = (global_basic_rank / m_total) * 100.0
      mcdc_exam = (global_mcdc_rank / m_total) * 100.0
      exam_impr = basic_exam - mcdc_exam

      return pd.Series([
          round(global_basic_rank, 2),
          round(global_mcdc_rank, 2),
          round(rank_impr, 2),
          round(basic_exam, 2),
          round(mcdc_exam, 2),
          round(exam_impr, 2),
          outcome,
      ])

    data[
        [
            f'{metric}_Global_Basic_Rank',
            f'{metric}_Global_MCDC_Rank',
            f'{metric}_Global_Rank_Impr',
            f'{metric}_Basic_EXAM_%',
            f'{metric}_MCDC_EXAM_%',
            f'{metric}_EXAM_Impr_%',
            f'{metric}_Outcome',
        ]
    ] = data.apply(calculate_global_ranks_and_outcome, axis=1)

  return data


# Compute global evaluation
df_global = evaluate_complete_sbfl(df)
# DISPLAY OUTPUT TABLE IN NOTEBOOK
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Display the top rows or the full dataframe
df_global

,Subject,Mutant,Failed_TCs,Total_Probes,Basic Tarantula,Basic Ochiai,Basic DStar,MC/DC Tarantula,MC/DC Ochiai,MC/DC DStar,Program_Total_Probes,Tarantula_Global_Basic_Rank,Tarantula_Global_MCDC_Rank,Tarantula_Global_Rank_Impr,Tarantula_Basic_EXAM_%,Tarantula_MCDC_EXAM_%,Tarantula_EXAM_Impr_%,Tarantula_Outcome,Ochiai_Global_Basic_Rank,Ochiai_Global_MCDC_Rank,Ochiai_Global_Rank_Impr,Ochiai_Basic_EXAM_%,Ochiai_MCDC_EXAM_%,Ochiai_EXAM_Impr_%,Ochiai_Outcome,DStar_Global_Basic_Rank,DStar_Global_MCDC_Rank,DStar_Global_Rank_Impr,DStar_Basic_EXAM_%,DStar_MCDC_EXAM_%,DStar_EXAM_Impr_%,DStar_Outcome
0,printtokens,v1,2,4,0.6828,0.0323,0.0021,0.7110,0.0345,0.0024,530,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN
1,printtokens,v2,26,4,0.6885,0.1175,0.3640,0.7321,0.1304,0.4501,530,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN
2,printtokens,v3,25,4,0.6904,0.1157,0.3395,0.7313,0.1277,0.4145,530,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN
3,printtokens,v4,11,4,0.6447,0.0628,0.0436,0.6904,0.0696,0.0535,530,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN
4,printtokens,v5,108,4,0.6958,0.2406,6.6348,0.7843,0.2983,10.5461,530,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN,2.5,1.0,1.5,0.47,0.19,0.28,WIN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,totinfo,v5,29,2,0.5536,0.1843,1.0194,0.5655,0.1886,1.0700,340,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN
91,totinfo,v6,41,2,0.4493,0.1452,0.8783,0.4615,0.1487,0.9216,340,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN
92,totinfo,v7,123,2,0.5596,0.3795,20.6963,0.5731,0.3885,21.8627,340,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN
93,totinfo,v8,199,2,0.5656,0.4827,60.4595,0.5807,0.4941,64.2873,340,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN,1.5,1.0,0.5,0.44,0.29,0.15,WIN
